# Vector DB Indexing & Similarity Search

Two questions this notebook answers with runnable code instead of just theory:

1. **Which distance/similarity metric should I use** (cosine, dot product, Euclidean, Manhattan, Jaccard) — and when do they actually give different rankings?
2. **Which index structure should my vector DB use** (flat, IVF, HNSW, LSH, product quantization, Annoy) — and what do I actually trade away for speed at scale?

Part 1 uses a small real sentence corpus (embedded with sentence-transformers) so metric differences are easy to reason about. Part 2 switches to a larger synthetic vector set (10k vectors) so build time, query latency, and **recall@10** (measured against brute-force ground truth) are meaningful — these differences barely show up at the scale of Part 1.


In [ ]:
%pip install -q numpy pandas scikit-learn sentence-transformers
%pip install -q faiss-cpu hnswlib annoy


## Part 1: Similarity Metrics

### Demo corpus

12 sentences, embedded once with `all-MiniLM-L6-v2`, reused for every metric below.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

demo_corpus = [
    "Retrieval-augmented generation combines a retriever with a language model.",   # 0
    "The system fetches relevant passages from a knowledge store before answering.",  # 1 - related to 0, few shared words
    "A vector database stores embeddings for fast similarity search.",              # 2
    "HNSW builds a multi-layer graph to search approximate nearest neighbors quickly.",  # 3
    "Product quantization compresses vectors to save memory at some cost to accuracy.",  # 4
    "The inverted file index clusters vectors so a search only scans a few cells.",  # 5
    "Cosine similarity measures the angle between two vectors, ignoring magnitude.", # 6
    "Euclidean distance measures the straight-line distance between two points.",    # 7
    "Locality-sensitive hashing places similar vectors into the same hash bucket.",  # 8
    "Annoy builds a forest of random-projection trees for approximate search.",      # 9
    "Recall at k measures how many true nearest neighbors an approximate method finds.",  # 10
    "The cat sat on the warm windowsill in the afternoon sun.",                      # 11 - unrelated
]

QUERY_IDX = 1  # "the system fetches relevant passages..." — used as the query for every metric demo below

embedder = SentenceTransformer("all-MiniLM-L6-v2")
demo_vectors = embedder.encode(demo_corpus, normalize_embeddings=True)  # unit-normalized — matters below
print(f"demo_vectors shape: {demo_vectors.shape}")


### Cosine Similarity

**What it is:** the cosine of the angle between two vectors — `dot(a, b) / (‖a‖ · ‖b‖)`. Ranges from -1 (opposite) to 1 (identical direction), and crucially **ignores magnitude**, only direction. This is the default metric for text embeddings because embedding *magnitude* mostly reflects things like sentence length or word frequency, not meaning — direction is where the semantic signal lives.

**When to use it:** the default choice for dense text embeddings, full stop — virtually every embedding model (SBERT, OpenAI, Vertex AI, Cohere) is trained/evaluated with cosine similarity in mind.


In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a_norm @ b_norm.T

query_vec = demo_vectors[QUERY_IDX]
cosine_scores = cosine_similarity(query_vec[None, :], demo_vectors)[0]
cosine_ranking = np.argsort(cosine_scores)[::-1]

print(f"query: {demo_corpus[QUERY_IDX]!r}\n")
for idx in cosine_ranking[:4]:
    print(f"  cos={cosine_scores[idx]:.3f} | [{idx}] {demo_corpus[idx]}")


### Dot Product (Inner Product)

**What it is:** `sum(a_i * b_i)` — no normalization step. Unlike cosine, this is sensitive to magnitude: a longer vector scores higher even in the same direction. **Important practical fact:** if every vector is already unit-normalized (as ours are, and as most embedding APIs return by default or on request), dot product and cosine similarity produce the *identical ranking* — this is exactly why FAISS's `IndexFlatIP` (inner product) is the standard way to do "cosine search": normalize once at index time, then use the cheaper dot product at query time.

**When to use it:** whenever vectors are pre-normalized — it's mathematically identical to cosine but skips the division, which matters at index-query scale.


In [ ]:
dot_scores = demo_vectors @ query_vec
dot_ranking = np.argsort(dot_scores)[::-1]

rankings_match = np.array_equal(cosine_ranking, dot_ranking)
print(f"dot-product ranking == cosine ranking (vectors are normalized)? {rankings_match}")
for idx in dot_ranking[:4]:
    print(f"  dot={dot_scores[idx]:.3f} | [{idx}] {demo_corpus[idx]}")


### Euclidean (L2) Distance

**What it is:** straight-line distance between two points, `‖a - b‖`. For **unit-normalized** vectors there's a direct algebraic relationship to cosine similarity: `‖a - b‖² = 2 - 2·cos(a, b)` — so ranking by ascending L2 distance is *mathematically guaranteed* to match ranking by descending cosine similarity on normalized vectors. This is why `IndexFlatL2` and `IndexFlatIP` give the same nearest neighbors once vectors are normalized, even though the raw numbers look completely different.

**When to use it:** interchangeable with cosine on normalized embeddings (pick whichever your vector DB/library makes cheaper); genuinely different — and usually the right choice — when vectors are *not* normalized and magnitude is meaningful (e.g. some non-text embedding spaces, spatial/coordinate data).


In [ ]:
l2_distances = np.linalg.norm(demo_vectors - query_vec, axis=1)
l2_ranking = np.argsort(l2_distances)  # ascending — smaller distance = more similar

rankings_match = np.array_equal(cosine_ranking, l2_ranking)
print(f"L2 ranking == cosine ranking (vectors are normalized)? {rankings_match}")

# verify the algebraic identity directly: ||a-b||^2 == 2 - 2*cos(a,b) for unit vectors
lhs = l2_distances[0] ** 2
rhs = 2 - 2 * cosine_scores[0]
print(f"||a-b||^2 = {lhs:.4f}   2 - 2*cos(a,b) = {rhs:.4f}   (should match)")


### Manhattan (L1) Distance

**What it is:** sum of absolute per-dimension differences, `sum(|a_i - b_i|)` — the "taxicab" distance. Unlike L2, there's **no equivalence to cosine even on normalized vectors** — L1 weighs many small differences differently than a few large ones, so it can (and does, below) produce a different ranking.

**When to use it:** rarely the first choice for text embeddings; more relevant in high-dimensional spaces prone to the "curse of dimensionality" where some research suggests L1 degrades more gracefully than L2, or when individual dimensions are independently meaningful (not the case for opaque learned embedding dimensions).


In [ ]:
l1_distances = np.abs(demo_vectors - query_vec).sum(axis=1)
l1_ranking = np.argsort(l1_distances)

rankings_match = np.array_equal(cosine_ranking, l1_ranking)
print(f"L1 ranking == cosine ranking? {rankings_match}  (not guaranteed, unlike L2)")
for idx in l1_ranking[:4]:
    print(f"  L1={l1_distances[idx]:.3f} | [{idx}] {demo_corpus[idx]}")


### Jaccard Similarity

**What it is:** `|A ∩ B| / |A ∪ B|` — overlap between two *sets*, not vectors in continuous space. Applied to text via a binary "does this word appear at all" representation (not counts, not weights). This is a **lexical** metric — it can only ever detect shared vocabulary, never paraphrase or synonymy — shown here specifically to contrast against the semantic metrics above.

**When to use it:** exact-match/dedup detection (near-duplicate documents, tag/category overlap), and as the theoretical basis for MinHash LSH (used for large-scale deduplication); essentially never used directly for semantic retrieval.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

binary_vectorizer = CountVectorizer(binary=True)
binary_matrix = binary_vectorizer.fit_transform(demo_corpus).toarray().astype(bool)

def jaccard_similarity(a: np.ndarray, b: np.ndarray) -> float:
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / union if union else 0.0

jaccard_scores = np.array([jaccard_similarity(binary_matrix[QUERY_IDX], binary_matrix[i]) for i in range(len(demo_corpus))])
jaccard_ranking = np.argsort(jaccard_scores)[::-1]

print("Jaccard's top matches (lexical overlap only — notice it misses [0], the semantically closest sentence):")
for idx in jaccard_ranking[:4]:
    print(f"  jaccard={jaccard_scores[idx]:.3f} | [{idx}] {demo_corpus[idx]}")


### Metrics Summary

| Metric | Sensitive to magnitude | Equivalent to cosine (on normalized vectors) | Typical use |
|---|---|---|---|
| Cosine similarity | no | — (this is the reference) | default for text embeddings |
| Dot product | yes (identical to cosine only if pre-normalized) | yes, if normalized | FAISS `IndexFlatIP` after normalizing at index time |
| Euclidean (L2) | yes (identical ranking to cosine only if pre-normalized) | yes, if normalized | FAISS/most ANN libraries' native default distance |
| Manhattan (L1) | yes | no | rarely used for embeddings; some high-dim robustness use cases |
| Jaccard | n/a (set-based, not vector) | no — lexical, not semantic | dedup, tag overlap, MinHash LSH |

**Practical upshot:** if your vectors are unit-normalized (most embedding APIs support this, and sentence-transformers does it via `normalize_embeddings=True`), cosine/dot-product/L2 are interchangeable for ranking purposes — pick whichever your index library computes fastest. Don't reach for L1 or Jaccard for semantic retrieval; they solve different problems.


## Part 2: Indexing Methods for Vector Databases

Every method below answers the same question — "given a query vector, find its nearest neighbors among N stored vectors" — but with different tradeoffs between **build time**, **query latency**, **recall** (how often it actually finds the true nearest neighbors), and **memory**. Brute-force (flat) is exact but scans everything; everything else is an **approximate nearest neighbor (ANN)** method that trades a small amount of recall for a large speedup.

### Benchmark setup

10,000 random unit vectors in 128 dimensions, 100 query vectors, `recall@10` measured against brute-force ground truth. These are synthetic (uniformly random on the unit sphere) rather than real embeddings — this is deliberately a *harder* case for ANN methods than real text embeddings, which cluster into dense semantic neighborhoods that approximate methods exploit well. Treat the recall numbers below as a lower bound, not what you'd see on your actual corpus.


In [ ]:
import time

rng = np.random.default_rng(42)
N, DIM = 10_000, 128
N_QUERIES, TOP_K = 100, 10

benchmark_vectors = rng.normal(size=(N, DIM)).astype("float32")
benchmark_vectors /= np.linalg.norm(benchmark_vectors, axis=1, keepdims=True)

benchmark_queries = rng.normal(size=(N_QUERIES, DIM)).astype("float32")
benchmark_queries /= np.linalg.norm(benchmark_queries, axis=1, keepdims=True)


def pad_to_k(ids: np.ndarray, k: int) -> np.ndarray:
    if len(ids) >= k:
        return np.asarray(ids[:k])
    return np.concatenate([ids, -1 * np.ones(k - len(ids), dtype=int)])


def recall_at_k(retrieved: np.ndarray, ground_truth: np.ndarray, k: int = TOP_K) -> float:
    hits = [len(set(retrieved[i][:k]) & set(ground_truth[i][:k])) / k for i in range(len(ground_truth))]
    return float(np.mean(hits))


benchmark_rows = []  # filled in by every section below: {"method": ..., "build_s": ..., "query_ms": ..., "recall@10": ..., "memory_mb": ...}
print(f"{N} vectors x {DIM} dims, {N_QUERIES} queries, top-{TOP_K}")


### Flat / Brute-Force (Exact)

**What it is:** compare the query against every single stored vector, no shortcuts — `O(N)` per query. Guaranteed to find the true nearest neighbors (recall = 1.0 by definition), which is exactly why it's used as the ground truth for every other method's recall measurement below.

**When to use it:** small collections (up to roughly tens of thousands of vectors, hardware-dependent) where 100% recall matters more than raw speed, or as a correctness baseline during development.


In [ ]:
import faiss

flat_index = faiss.IndexFlatIP(DIM)  # inner product on unit vectors == cosine ranking (see Part 1)

start = time.perf_counter()
flat_index.add(benchmark_vectors)
flat_build_s = time.perf_counter() - start

start = time.perf_counter()
_, ground_truth_ids = flat_index.search(benchmark_queries, TOP_K)
flat_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

flat_memory_mb = N * DIM * 4 / 1e6  # raw float32 vectors

benchmark_rows.append({"method": "flat (exact)", "build_s": flat_build_s, "query_ms": flat_query_ms,
                        "recall@10": 1.0, "memory_mb": flat_memory_mb})
print(f"flat: build={flat_build_s:.3f}s, avg query={flat_query_ms:.3f}ms, memory={flat_memory_mb:.1f}MB")


### IVF (Inverted File Index)

**What it is:** run k-means once to partition the vector space into `nlist` clusters ("Voronoi cells"), then at query time only scan the `nprobe` cells nearest to the query instead of the whole dataset — an `O(N/nlist * nprobe)` search instead of `O(N)`. `nprobe` is the key recall/speed knob: `nprobe=nlist` degenerates back to brute force; `nprobe=1` is fastest but can miss neighbors sitting just across a cell boundary.

**When to use it:** large collections where flat is too slow, and you can afford a training step (k-means over a representative sample) before indexing — the classic "medium-to-large scale" choice, often paired with PQ (below) for very large scale.


In [ ]:
nlist = 100  # number of clusters — rule of thumb: roughly sqrt(N) to 4*sqrt(N)
quantizer = faiss.IndexFlatIP(DIM)
ivf_index = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)

start = time.perf_counter()
ivf_index.train(benchmark_vectors)
ivf_index.add(benchmark_vectors)
ivf_build_s = time.perf_counter() - start

ivf_index.nprobe = 8  # scan 8 of 100 cells — tune this for your recall/latency target

start = time.perf_counter()
_, ivf_ids = ivf_index.search(benchmark_queries, TOP_K)
ivf_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

ivf_recall = recall_at_k(ivf_ids, ground_truth_ids)
benchmark_rows.append({"method": f"IVF (nlist={nlist}, nprobe=8)", "build_s": ivf_build_s, "query_ms": ivf_query_ms,
                        "recall@10": ivf_recall, "memory_mb": N * DIM * 4 / 1e6})
print(f"IVF: build={ivf_build_s:.3f}s, avg query={ivf_query_ms:.3f}ms, recall@10={ivf_recall:.3f}")


### HNSW (Hierarchical Navigable Small World)

**What it is:** builds a multi-layer graph where each vector is a node connected to its approximate nearest neighbors — sparse long-range links on upper layers for fast coarse navigation, dense short-range links on the bottom layer for precision. A query greedily walks the graph, descending layers, homing in on the true neighborhood in roughly `O(log N)` hops. No training step (unlike IVF/PQ) — vectors are added incrementally.

**When to use it:** the most common default in production vector databases today (Chroma, Weaviate, Qdrant, and pgvector's HNSW index all use this) — excellent recall/speed tradeoff without a training phase, at the cost of higher memory (storing the graph edges) and slower builds/inserts than flat.


In [ ]:
import hnswlib

M_PARAM, EF_CONSTRUCTION, EF_SEARCH = 16, 200, 50  # M = graph connectivity, ef = candidate-list size (build/search)

hnsw_index = hnswlib.Index(space="ip", dim=DIM)

start = time.perf_counter()
hnsw_index.init_index(max_elements=N, ef_construction=EF_CONSTRUCTION, M=M_PARAM)
hnsw_index.add_items(benchmark_vectors, np.arange(N))
hnsw_build_s = time.perf_counter() - start

hnsw_index.set_ef(EF_SEARCH)  # higher ef_search = better recall, slower query — the main runtime knob

start = time.perf_counter()
hnsw_ids, _ = hnsw_index.knn_query(benchmark_queries, k=TOP_K)
hnsw_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

hnsw_recall = recall_at_k(hnsw_ids, ground_truth_ids)
# rough memory estimate: raw vectors + ~M links/node * 2 directions * 4 bytes (int32 id) per layer-0 edge
hnsw_memory_mb = (N * DIM * 4 + N * M_PARAM * 2 * 4) / 1e6
benchmark_rows.append({"method": f"HNSW (M={M_PARAM}, ef={EF_SEARCH})", "build_s": hnsw_build_s, "query_ms": hnsw_query_ms,
                        "recall@10": hnsw_recall, "memory_mb": hnsw_memory_mb})
print(f"HNSW: build={hnsw_build_s:.3f}s, avg query={hnsw_query_ms:.3f}ms, recall@10={hnsw_recall:.3f}")


### LSH (Locality-Sensitive Hashing)

**What it is:** hash each vector with several random hyperplanes — which side of each hyperplane a vector falls on becomes one bit of its hash. Vectors that are close together tend to land on the same side of most hyperplanes, so they collide into the same bucket with high probability; a query only needs to check vectors sharing its bucket, then exactly reranks that small candidate set. Multiple independent hash tables (`n_tables`) trade memory for recall — more tables catch neighbors a single unlucky hash missed.

**When to use it:** historically important (predates HNSW as the standard ANN approach) and still used in specific settings like MinHash-based deduplication or when hash-based sharding fits an existing infrastructure — largely superseded by HNSW/IVF for general-purpose vector search, shown here mainly so the mechanism (and why graph-based methods overtook it) is concrete rather than a black box.


In [ ]:
class RandomHyperplaneLSH:
    def __init__(self, dim: int, n_bits: int = 16, n_tables: int = 6, seed: int = 42):
        rng = np.random.default_rng(seed)
        self.n_tables = n_tables
        self.hyperplanes = [rng.normal(size=(n_bits, dim)) for _ in range(n_tables)]
        self.tables: list[dict] = [dict() for _ in range(n_tables)]
        self.vectors = None

    def _hash(self, vec: np.ndarray, table_idx: int) -> tuple:
        return tuple((self.hyperplanes[table_idx] @ vec > 0).astype(int))

    def build(self, vectors: np.ndarray) -> None:
        self.vectors = vectors
        for table_idx in range(self.n_tables):
            for i, vec in enumerate(vectors):
                key = self._hash(vec, table_idx)
                self.tables[table_idx].setdefault(key, []).append(i)

    def search(self, query: np.ndarray, k: int = 10) -> np.ndarray:
        candidates: set = set()
        for table_idx in range(self.n_tables):
            candidates.update(self.tables[table_idx].get(self._hash(query, table_idx), []))
        if not candidates:
            return np.array([], dtype=int)
        candidate_ids = np.array(list(candidates))
        sims = self.vectors[candidate_ids] @ query  # exact rerank within the (small) hashed candidate set
        return candidate_ids[np.argsort(sims)[::-1][:k]]


lsh_index = RandomHyperplaneLSH(DIM, n_bits=16, n_tables=6)

start = time.perf_counter()
lsh_index.build(benchmark_vectors)
lsh_build_s = time.perf_counter() - start

start = time.perf_counter()
lsh_ids = np.stack([pad_to_k(lsh_index.search(q, TOP_K), TOP_K) for q in benchmark_queries])
lsh_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

lsh_recall = recall_at_k(lsh_ids, ground_truth_ids)
lsh_memory_mb = N * DIM * 4 / 1e6  # LSH here still stores full vectors for the exact-rerank step
benchmark_rows.append({"method": "LSH (random hyperplane)", "build_s": lsh_build_s, "query_ms": lsh_query_ms,
                        "recall@10": lsh_recall, "memory_mb": lsh_memory_mb})
print(f"LSH: build={lsh_build_s:.3f}s, avg query={lsh_query_ms:.3f}ms, recall@10={lsh_recall:.3f}")


### Product Quantization (PQ / IVFPQ)

**What it is:** split each vector into `m` sub-vectors, and independently vector-quantize each sub-vector to one of `2^nbits` learned centroids (a small per-subspace codebook) — a 128-dim float32 vector (512 bytes) becomes `m` tiny integer codes (e.g. 16 bytes for m=16, nbits=8). Distances are then computed approximately from the codes via precomputed lookup tables, never fully reconstructing the original vector. This is a **memory/compression** technique first, a speed technique second — typically combined with IVF (as `IndexIVFPQ`) so a query narrows to a few clusters *and* scans compressed codes within them.

**When to use it:** billion-scale or memory-constrained deployments where storing full-precision vectors for every item is the actual bottleneck, not query latency — the standard technique behind "how does anyone search a billion embeddings without a terabyte of RAM."


In [ ]:
m_subquantizers, nbits = 16, 8  # DIM=128 must be divisible by m_subquantizers (128 / 16 = 8 dims/subvector)
pq_quantizer = faiss.IndexFlatIP(DIM)
pq_index = faiss.IndexIVFPQ(pq_quantizer, DIM, nlist, m_subquantizers, nbits, faiss.METRIC_INNER_PRODUCT)

start = time.perf_counter()
pq_index.train(benchmark_vectors)
pq_index.add(benchmark_vectors)
pq_build_s = time.perf_counter() - start

pq_index.nprobe = 8
start = time.perf_counter()
_, pq_ids = pq_index.search(benchmark_queries, TOP_K)
pq_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

pq_recall = recall_at_k(pq_ids, ground_truth_ids)
pq_memory_mb = N * m_subquantizers * (nbits / 8) / 1e6  # compressed codes only
benchmark_rows.append({"method": f"IVFPQ (m={m_subquantizers}, nbits={nbits})", "build_s": pq_build_s, "query_ms": pq_query_ms,
                        "recall@10": pq_recall, "memory_mb": pq_memory_mb})
print(f"IVFPQ: build={pq_build_s:.3f}s, avg query={pq_query_ms:.3f}ms, recall@10={pq_recall:.3f}")
print(f"memory: flat={flat_memory_mb:.1f}MB vs IVFPQ={pq_memory_mb:.1f}MB ({flat_memory_mb / pq_memory_mb:.0f}x smaller)")


### Annoy (Approximate Nearest Neighbors Oh Yeah)

**What it is:** builds a forest of binary trees, each constructed by recursively splitting the vector space with random hyperplanes; a query descends each tree to a leaf, collects candidates from all trees, and reranks exactly. Popularized by Spotify for music recommendation. Its defining feature is that a built index can be memory-mapped straight from disk — many processes can share one index's memory without each loading a full copy, which HNSW's graph structure doesn't support as cleanly.

**When to use it:** read-heavy, mostly-static indexes (build once, query many times, rebuild rarely) where memory-mapping a shared index file across worker processes matters more than fast incremental inserts — HNSW is generally preferred when the index needs frequent updates.


In [ ]:
from annoy import AnnoyIndex

n_trees = 10  # more trees = better recall, larger index, slower build
annoy_index = AnnoyIndex(DIM, "dot")

start = time.perf_counter()
for i, vec in enumerate(benchmark_vectors):
    annoy_index.add_item(i, vec)
annoy_index.build(n_trees)
annoy_build_s = time.perf_counter() - start

start = time.perf_counter()
annoy_ids = np.array([annoy_index.get_nns_by_vector(q, TOP_K) for q in benchmark_queries])
annoy_query_ms = (time.perf_counter() - start) / N_QUERIES * 1000

annoy_recall = recall_at_k(annoy_ids, ground_truth_ids)
annoy_memory_mb = N * DIM * 4 * (1 + n_trees * 0.1) / 1e6  # rough — trees add overhead beyond raw vectors
benchmark_rows.append({"method": f"Annoy (n_trees={n_trees})", "build_s": annoy_build_s, "query_ms": annoy_query_ms,
                        "recall@10": annoy_recall, "memory_mb": annoy_memory_mb})
print(f"Annoy: build={annoy_build_s:.3f}s, avg query={annoy_query_ms:.3f}ms, recall@10={annoy_recall:.3f}")


### Benchmark Comparison


In [ ]:
import pandas as pd

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df["build_s"] = benchmark_df["build_s"].round(3)
benchmark_df["query_ms"] = benchmark_df["query_ms"].round(3)
benchmark_df["recall@10"] = benchmark_df["recall@10"].round(3)
benchmark_df["memory_mb"] = benchmark_df["memory_mb"].round(2)
benchmark_df.sort_values("query_ms").reset_index(drop=True)


## Production Vector Databases — What They Actually Use

Every technique above is the engine under the hood of a real vector database, not just an academic exercise:

| Database | Default/primary index | Notes |
|---|---|---|
| Chroma | HNSW (via `hnswlib`) | exactly the library used in Part 2 — `vertex_rag_pipeline.ipynb` in this folder has been building HNSW indexes all along |
| Weaviate | HNSW | tunable `ef`/`efConstruction`, same knobs as above |
| Qdrant | HNSW | with optional scalar/product quantization layered on top for memory |
| Pinecone | proprietary graph-based (HNSW-like) | managed, no index-type choice exposed |
| Milvus | IVF, HNSW, DiskANN, or Annoy | the one major vector DB that lets you pick the index family per collection |
| pgvector (Postgres) | IVFFlat or HNSW | a Postgres extension — the only option here that isn't a dedicated vector DB |
| FAISS | not a database — a library | what several of the above use *internally*; you built flat/IVF/HNSW/PQ indexes with it directly in this notebook |

## Summary

- **Metrics**: on unit-normalized embeddings, cosine similarity, dot product, and Euclidean distance all produce the *same ranking* — use whichever your library computes natively (usually dot product or L2). Manhattan and Jaccard solve different problems and aren't drop-in substitutes for semantic search.
- **Indexing**: flat is exact but `O(N)` — fine until it isn't. HNSW is the strongest general-purpose default (used by most production vector DBs for exactly this reason). IVF is a good large-scale choice when you can afford a training step. PQ (usually as IVFPQ) is what you reach for when memory, not latency, is the bottleneck. LSH is worth understanding but rarely the first choice today. Annoy earns its place specifically for shared, mostly-static, memory-mapped indexes.
- **Tuning knob to remember**: every ANN method trades recall for speed through one main parameter — `nprobe` (IVF/PQ), `ef`/`ef_construction` (HNSW), `n_tables`/`n_bits` (LSH), `n_trees` (Annoy). None of these numbers are universal; benchmark on your real corpus and query distribution the way this notebook benchmarked on synthetic data, using recall@k against a brute-force baseline as the ground truth.
